In [10]:
# Load ChemBERTa model and tokenizer for drug feature extraction
from transformers import AutoTokenizer, AutoModel
import torch

# Initialize the tokenizer and model for ChemBERTa
chem_tokenizer = AutoTokenizer.from_pretrained("seyonec/ChemBERTa-zinc-base-v1")
chem_model = AutoModel.from_pretrained("seyonec/ChemBERTa-zinc-base-v1")

In [ ]:
from transformers import AutoTokenizer, AutoModel

model_path = r"D:\Drugllm\esm2_t6_8M_UR50D"

esm_tokenizer = AutoTokenizer.from_pretrained(model_path)
esm_model = AutoModel.from_pretrained(model_path)

esm_model.eval()  # important for feature extraction

<>:5: SyntaxWarning: invalid escape sequence '\D'
<>:6: SyntaxWarning: invalid escape sequence '\D'
<>:5: SyntaxWarning: invalid escape sequence '\D'
<>:6: SyntaxWarning: invalid escape sequence '\D'
C:\Users\Cindy\AppData\Local\Temp\ipykernel_16652\1059658157.py:5: SyntaxWarning: invalid escape sequence '\D'
  esm_tokenizer = AutoTokenizer.from_pretrained("D:\Drugllm\esm2_t6_8M_UR50D")
C:\Users\Cindy\AppData\Local\Temp\ipykernel_16652\1059658157.py:6: SyntaxWarning: invalid escape sequence '\D'
  esm_model = AutoModel.from_pretrained("D:\Drugllm\esm2_t6_8M_UR50D")
Some weights of EsmModel were not initialized from the model checkpoint at D:\Drugllm\esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
# Print the hidden size of the ChemBERTa model
print(chem_model.config.hidden_size)

768


In [13]:
# Print the hidden size of the ESM2 model
print(esm_model.config.hidden_size)

320


In [14]:
# Define functions to extract features from drugs and proteins using pre-trained models
def extract_chem_features(smiles):
    """Extract ChemBERTa features from SMILES strings."""
    try:
        # Tokenize the SMILES string
        tokens = chem_tokenizer(smiles, return_tensors="pt", padding=True, truncation=True)
        # Generate embeddings using the ChemBERTa model
        with torch.no_grad():
            embeddings = chem_model(**tokens).last_hidden_state.mean(dim=1).squeeze().numpy()
        return embeddings
    except:
        # Return a zero vector if feature extraction fails
        return np.zeros(768)

def extract_esm_features(sequence):
    """Extract ESM2 features from protein sequences."""
    try:
        # Tokenize the protein sequence
        tokens = esm_tokenizer(sequence, return_tensors="pt", padding=True, truncation=True)
        # Generate embeddings using the ESM2 model
        with torch.no_grad():
            embeddings = esm_model(**tokens).last_hidden_state.mean(dim=1).squeeze().numpy()
        return embeddings
    except:
        # Return a zero vector if feature extraction fails
        return np.zeros(320)

In [15]:
import pandas as pd
pkd_drug_candidates = pd.read_csv("D:\\Github\\llm-drug-agent\\PKDisease\\drug_target_cross_join_01042026.csv")
pkd_drug_candidates.head()

,Chem ID,Smiles,Drug Name,maximumClinicalTrialPhase,isApproved,targetId,score,Gene,priority,Entry,Entry Name,Gene Names,Length,Sequence,EC number,Uniprot,known_pair,status
0,CHEMBL107201,N=C(N)SCCc1cccc(CCSC(=N)N)c1,CHEMBL107201,NaN,NaN,ENSG00000005381,0.158737,MPO,0.593478,P05164,PERM_HUMAN,MPO,745.0,MGVPFFSSLRCMVDLGPCWAGGLTAEMKLLLALAGLLAILATPQPS...,1.11.2.2,P05164,"('CHEMBL107201', 'ENSG00000005381')",False
1,CHEMBL107201,N=C(N)SCCc1cccc(CCSC(=N)N)c1,CHEMBL107201,NaN,NaN,ENSG00000006071,0.060793,ABCC8,0.567669,Q09428,ABCC8_HUMAN,ABCC8 HRINS SUR SUR1,1581.0,MPLAFCGSENHSAAYRVDQGVLNNGCFVDALNVVPHVFLLFITFPI...,NaN,Q09428,"('CHEMBL107201', 'ENSG00000006071')",False
2,CHEMBL107201,N=C(N)SCCc1cccc(CCSC(=N)N)c1,CHEMBL107201,NaN,NaN,ENSG00000007314,0.655636,SCN4A,0.545397,P35499,SCN4A_HUMAN,SCN4A,1836.0,MARPSLCTLVPLGPECLRPFTRESLAAIEQRAVEEEARLQRNKQME...,NaN,P35499,"('CHEMBL107201', 'ENSG00000007314')",False
3,CHEMBL107201,N=C(N)SCCc1cccc(CCSC(=N)N)c1,CHEMBL107201,NaN,NaN,ENSG00000011677,0.174653,GABRA3,0.537177,P34903,GBRA3_HUMAN,GABRA3,492.0,MIITQTSHCYMTSLGILFLINILPGTTGQGESRRQEPGDFVKQDIG...,NaN,P34903,"('CHEMBL107201', 'ENSG00000011677')",False
4,CHEMBL107201,N=C(N)SCCc1cccc(CCSC(=N)N)c1,CHEMBL107201,NaN,NaN,ENSG00000012504,0.136784,NR1H4,0.537302,Q96RI1,NR1H4_HUMAN,NR1H4 BAR FXR HRR1 RIP14,486.0,MVMQFQGLENPIQISPHCSCTPSGFFMEMMSMKPAKGVLTEQVAGP...,NaN,Q96RI1,"('CHEMBL107201', 'ENSG00000012504')",False


In [16]:
# Featurize drugs
# Extract unique drugs and proteins
unique_drugs = pkd_drug_candidates[['Smiles']].drop_duplicates()
unique_proteins = pkd_drug_candidates[['Sequence']].drop_duplicates()
# Import tqdm for progress bars during feature extraction
from tqdm import tqdm
# Extract features for unique drugs in the BindDB dataset
tqdm.pandas()  # Enable progress bar for pandas operations
unique_drugs['drug_features'] = unique_drugs['Smiles'].progress_apply(extract_chem_features)
# Extract features for unique proteins in the BindDB dataset
unique_proteins['protein_features'] = unique_proteins['Sequence'].progress_apply(extract_esm_features)
# Merge extracted features back into the BindDB dataset
pkd_drug_candidates = pkd_drug_candidates.merge(unique_drugs, on='Smiles', how='left')
pkd_drug_candidates = pkd_drug_candidates.merge(unique_proteins, on='Sequence', how='left')
# Save the featurized BindDB dataset to a PyTorch file
torch.save(pkd_drug_candidates, "D:\\Drugllm\\PKDisease\\pkd_drug_candidates_featurized_v2.pt")

100%|██████████| 348/348 [02:41<00:00,  2.15it/s]


In [ ]:
# Featurize drugs
# Extract unique drugs and proteins
unique_drugs = bind_db[['Drug']].drop_duplicates()
unique_proteins = bind_db[['Target']].drop_duplicates()

In [ ]:
# Import tqdm for progress bars during feature extraction
from tqdm import tqdm

In [ ]:
# Extract features for unique drugs in the BindDB dataset
tqdm.pandas()  # Enable progress bar for pandas operations
unique_drugs['drug_features'] = unique_drugs['Drug'].progress_apply(extract_chem_features)

In [ ]:
# Extract features for unique proteins in the BindDB dataset
unique_proteins['protein_features'] = unique_proteins['Target'].progress_apply(extract_esm_features)

In [ ]:
# Merge extracted features back into the BindDB dataset
bind_db = bind_db.merge(unique_drugs, on='Drug', how='left')
bind_db = bind_db.merge(unique_proteins, on='Target', how='left')

In [ ]:
# Save the featurized BindDB dataset to a PyTorch file
torch.save(bind_db, '/content/drive/MyDrive/DrugPLM-Cindy-2025/Code_and_Data/Data/BindDB/BindDB_featurized.pt')

In [ ]:
# Extract unique drugs and proteins from the Davis dataset
unique_drugs = davis_db[['Drug']].drop_duplicates()
unique_proteins = davis_db[['Target']].drop_duplicates()

In [ ]:
# Extract features for unique drugs in the Davis dataset
tqdm.pandas()  # Enable progress bar for pandas operations
unique_drugs['drug_features'] = unique_drugs['Drug'].progress_apply(extract_chem_features)

In [ ]:
# Extract features for unique proteins in the Davis dataset
unique_proteins['protein_features'] = unique_proteins['Target'].progress_apply(extract_esm_features)

In [ ]:
# Merge extracted features back into the Davis dataset
davis_db = davis_db.merge(unique_drugs, on='Drug', how='left')
davis_db = davis_db.merge(unique_proteins, on='Target', how='left')

In [ ]:
# Save the featurized Davis dataset to a PyTorch file
torch.save(davis_db, '/content/drive/MyDrive/DrugPLM-Cindy-2025/Code_and_Data/Data/Davis/Davis_featurized.pt')

In [ ]:
# Extract unique drugs and proteins from the Kiba dataset
unique_drugs = kiba_db[['Drug']].drop_duplicates()
unique_proteins = kiba_db[['Target']].drop_duplicates()

In [ ]:
# Extract features for unique drugs in the Kiba dataset
tqdm.pandas()  # Enable progress bar for pandas operations
unique_drugs['drug_features'] = unique_drugs['Drug'].progress_apply(extract_chem_features)

In [ ]:
# Extract features for unique proteins in the Kiba dataset
unique_proteins['protein_features'] = unique_proteins['Target'].progress_apply(extract_esm_features)

In [ ]:
# Merge extracted features back into the Kiba dataset
kiba_db = kiba_db.merge(unique_drugs, on='Drug', how='left')
kiba_db = kiba_db.merge(unique_proteins, on='Target', how='left')

In [ ]:
# Save the featurized Kiba dataset to a PyTorch file
torch.save(kiba_db, '/content/drive/MyDrive/DrugPLM-Cindy-2025/Code_and_Data/Data/Kiba/Kiba_featurized.pt')

In [ ]:
# Load the featurized BindDB dataset for further analysis
bind_db = torch.load('/content/drive/MyDrive/DrugPLM-Cindy-2025/Code_and_Data/Data/BindDB/BindDB_featurized.pt', weights_only=False)
# Display the first few rows of the dataset
bind_db.head()